In [ ]:
# Load packages
import os.path as op
import glob
import shutil
import mne
import numpy as np
import pandas as pd

In [ ]:
# Make sub id list
subject_ids = 1
# Define data paths
root_dir = 'C:/Users/mvmigem/Documents/data/project_2/'
raw_data_dir = op.join(root_dir,"raw")
# get individual paths
sub_paths = glob.glob(raw_data_dir + "/sub*")

In [23]:
# iterate over subjects
subject = 1
sub_i = subject-1  
# EEG part
raw_path = op.join(sub_paths[sub_i],'eeg','*.bdf')
raw_fname = glob.glob(raw_path) [0]
raw = mne.io.read_raw_bdf(raw_fname, preload = False)
events = mne.find_events(raw)

# Behaviour part
behav_path = op.join(sub_paths[sub_i],'behav','*.csv')
behav_fname = glob.glob(behav_path) [0]
behav = pd.read_csv(behav_fname)

Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-01\eeg\main_01.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
4530 events found on stim channel Status
Event IDs: [   11    13    22    24    31    33    42    44    99 65536 65635 65789]


In [24]:
values_to_remove = [99,11,12,13,14,21,22,23,24,31,32,33,34,41,42,43,44]
events = events[np.isin(events[:, -1], values_to_remove)]

In [25]:
# Adjust the behaviour data to fit the event structure 
def create_header_row(original_row):
    """Create a header row based on the first row of a trial"""
    header_row = original_row.copy()
    header_row['eeg_trigger'] = 99
    header_row['t_stim'] = 0
    header_row['sequence'] = None
    header_row['position'] = None
    return header_row

new_rows = []

for i in range(0, len(behav), 4):
    # Create header row from first row of trial
    header_row = create_header_row(behav.iloc[i])
    new_rows.append(pd.DataFrame([header_row], index=[i - 0.5]))
    
    # Add original trial rows
    new_rows.append(behav.iloc[i:i+4])

result_df = pd.concat(new_rows).sort_index().reset_index(drop=True)

In [26]:
result_df = result_df.iloc[1:]

In [29]:
# there was an error that accidentaly dropped the last/first stims of  trials (prob due to block cuttoff)
lost_stim = []
shift_comp = 0
for i in range(len(events)-5):
    if events[i,2] == 99:
        if events[i +5,2] !=99:
            lost_stim.append(i+5+ shift_comp)
            shift_comp += 1
meta_data = result_df.drop(lost_stim).reset_index(drop=True)

In [32]:
check = events[:,2] == meta_data['eeg_trigger'].to_numpy()

In [31]:
meta_data= meta_data.iloc[:len(events[:,2])]

In [30]:
meta_data = meta_data.drop(1953)